# VECTOR HOÁ TF-IDF VÀ CÁC MÔ HÌNH BASELINE (V3)

In [1]:
import os
import re
import sys
import joblib
import pandas as pd
import numpy as np
from scipy.special import expit

In [2]:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
MODEL_DIR = os.path.join(ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
if not os.environ.get('TEMP'):
    os.environ['TEMP'] = os.path.join(ROOT, '.tmp_run')
os.makedirs(os.environ['TEMP'], exist_ok=True)
os.environ['TMP'] = os.environ['TEMP']

In [3]:
#Nap helper dung chung tu src/ de dong nhat chia Dev 3 phan, suy dien Multi-Label va cham True ABSA 5x4
from src.labels import (
    ABSA_COLS, ASPECT_GROUPS, classify_silent_complaint_levels,
    compute_absa_metrics, derive_overall_sentiment,
)
from src.modeling import split_dev_three_way, multilabel_to_primary_aspect


In [4]:
#Doc 3 tap du lieu sach tu buoc 01
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
df_test = pd.read_csv(os.path.join(PROC_DIR, 'test_clean.csv'))

In [5]:
df_train.shape, df_dev.shape, df_test.shape

((7762, 24), (1110, 24), (2224, 24))

In [6]:
asp_cols = ['asp_0', 'asp_1', 'asp_2', 'asp_3', 'asp_4']
df_train[['sentence_seg', 'sentiment', 'aspect_group'] + asp_cols].head(3)

,sentence_seg,sentiment,aspect_group,asp_0,asp_1,asp_2,asp_3,asp_4
0,Mới mua máy này Tại thegioididong thốt_nốt cảm...,2,2,1,1,1,1,1
1,Pin kém còn lại miễn chê mua 8 / 3 / 2019 tình...,1,2,0,0,1,0,1
2,Sao lúc gọi điện_thoại màn_hình bị chấm nhỏ nh...,0,0,1,0,0,0,0


## CHIA TẬP DEV 3 PHẦN ĐỘC LẬP (EYEBALL, CALIB, VAL)

In [7]:
#Chia dev thanh Eyeball (250), Calib (300), Val (560); tuyet doi khong dung Val khi tune CV
df_eyeball, df_calib, df_val = split_dev_three_way(df_dev, random_state=42)
df_eyeball.shape, df_calib.shape, df_val.shape

((250, 24), (300, 24), (560, 24))

In [8]:
pd.DataFrame({
    'eyeball': df_eyeball['sentiment'].value_counts(),
    'calib': df_calib['sentiment'].value_counts(),
    'val': df_val['sentiment'].value_counts(),
}).sort_index()

,eyeball,calib,val
sentiment,,,
0,78,96,176
1,48,58,110
2,124,146,274


In [9]:
pd.DataFrame({
    'eyeball': df_eyeball['aspect_group'].value_counts(),
    'calib': df_calib['aspect_group'].value_counts(),
    'val': df_val['aspect_group'].value_counts(),
}).sort_index()

,eyeball,calib,val
aspect_group,,,
0,104,125,234
1,32,38,72
2,53,63,118
3,33,40,73
4,28,34,63


## TRỌNG SỐ LỚP LÀM MỊN VÀ ĐẶC TRƯNG TF-IDF

In [10]:
#Tinh can bac hai nghich dao tan suat lop cho sentiment de chong overfit lop thieu so
s_cnt = df_train['sentiment'].value_counts().to_dict()
smooth_w_sent = {c: float(np.sqrt(len(df_train) / (3.0 * count))) for c, count in s_cnt.items()}
smooth_w_sent

{2: 0.833306494468503, 0: 1.0142784473314057, 1: 1.3042522205923242}

In [11]:
#Tinh trong so lam min cho 5 nhom khia canh
a_cnt = df_train['aspect_group'].value_counts().to_dict()
smooth_w_asp = {c: float(np.sqrt(len(df_train) / (5.0 * count))) for c, count in a_cnt.items()}
smooth_w_asp

{0: 0.6981473577433484,
 2: 0.9732229468163418,
 3: 1.1906791738235658,
 1: 1.2892239918147133,
 4: 1.3068323462013434}

In [12]:
X_tr_text = df_train['sentence_seg'].fillna('')
X_eye_text = df_eyeball['sentence_seg'].fillna('')
X_cal_text = df_calib['sentence_seg'].fillna('')
X_val_text = df_val['sentence_seg'].fillna('')
X_te_text = df_test['sentence_seg'].fillna('')

In [13]:
y_tr_sent, y_eye_sent, y_val_sent, y_te_sent = (
    df_train['sentiment'].values, df_eyeball['sentiment'].values,
    df_val['sentiment'].values, df_test['sentiment'].values
)
y_tr_asp, y_eye_asp, y_val_asp, y_te_asp = (
    df_train['aspect_group'].values, df_eyeball['aspect_group'].values,
    df_val['aspect_group'].values, df_test['aspect_group'].values
)

In [14]:
Y_tr_asp = df_train[asp_cols].values
Y_eye_asp = df_eyeball[asp_cols].values
Y_val_asp = df_val[asp_cols].values
Y_te_asp = df_test[asp_cols].values
Y_tr_asp.shape, Y_val_asp.shape, Y_te_asp.shape

((7762, 5), (560, 5), (2224, 5))

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [16]:
#Giu token 1 ky tu ('1'..'5', 'o', 'a', 'y') va PII tags ([PHONE], [ORDER_ID], [EMAIL], [URL], [PERSON])
tfidf_params = dict(
    token_pattern=r'(?u)\b\w+\b|\[\w+\]',
    ngram_range=(1, 2),
    max_features=15000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    lowercase=True,
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [17]:
#Chi fit TF-IDF doc lap tren df_train['sentence_seg'] de phuc vu SHAP/tuong thich, khong cham Dev/Test
vec = TfidfVectorizer(**tfidf_params)
X_tr_tfidf = vec.fit_transform(X_tr_text)
vocab_check = {t: (t in vec.vocabulary_) for t in ['1', '5', 'ở', 'ạ', 'ý', '[phone]', '[url]', '[person]']}
X_tr_tfidf.shape, vocab_check

((7762, 15000),
 {'1': True,
  '5': True,
  'ở': True,
  'ạ': True,
  'ý': True,
  '[phone]': True,
  '[url]': True,
  '[person]': True})

## TINH CHỈNH MÔ HÌNH CẢM XÚC BẰNG PIPELINE + STRATIFIEDKFOLD(5)

In [18]:
from sklearn.naive_bayes import ComplementNB
#1. ComplementNB trong Pipeline + StratifiedKFold(5) thuan tren df_train
pipe_nb = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', ComplementNB())])
nb_grid = {'clf__alpha': [0.05, 0.1, 0.25, 0.5, 1.0], 'clf__norm': [True, False]}
nb_search = GridSearchCV(pipe_nb, nb_grid, cv=skf, scoring='f1_macro', n_jobs=1, refit=True)
nb_search.fit(X_tr_text, y_tr_sent)
nb_sent_pipe = nb_search.best_estimator_
print('NB best params:', nb_search.best_params_)

NB best params: {'clf__alpha': 0.05, 'clf__norm': True}


In [19]:
f1_train = f1_score(y_tr_sent, nb_sent_pipe.predict(X_tr_text), average='macro')
f1_cv = float(nb_search.best_score_)
f1_eyeball = f1_score(y_eye_sent, nb_sent_pipe.predict(X_eye_text), average='macro')
f1_val = f1_score(y_val_sent, nb_sent_pipe.predict(X_val_text), average='macro')
print(f'NB Sent -> Train F1: {f1_train:.4f} | CV F1: {f1_cv:.4f} | Eyeball F1: {f1_eyeball:.4f} | Val F1: {f1_val:.4f}')
nb_sent_scores = {'model': 'ComplementNB', 'pipe': nb_sent_pipe, 'train_f1': f1_train, 'cv_f1': f1_cv, 'eyeball_f1': f1_eyeball, 'val_f1': f1_val}

NB Sent -> Train F1: 0.8889 | CV F1: 0.7024 | Eyeball F1: 0.6924 | Val F1: 0.7240


In [20]:
from sklearn.linear_model import LogisticRegression
#2. LogisticRegression trong Pipeline + StratifiedKFold(5) thuan tren df_train
pipe_lr = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
lr_grid = {'clf__C': [0.2, 0.5, 1.0, 2.0, 5.0], 'clf__class_weight': [None, 'balanced', smooth_w_sent]}
lr_search = GridSearchCV(pipe_lr, lr_grid, cv=skf, scoring='f1_macro', n_jobs=1, refit=True)
lr_search.fit(X_tr_text, y_tr_sent)
lr_sent_pipe = lr_search.best_estimator_
print('LR best params:', lr_search.best_params_)

LR best params: {'clf__C': 2.0, 'clf__class_weight': 'balanced'}


In [21]:
f1_train = f1_score(y_tr_sent, lr_sent_pipe.predict(X_tr_text), average='macro')
f1_cv = float(lr_search.best_score_)
f1_eyeball = f1_score(y_eye_sent, lr_sent_pipe.predict(X_eye_text), average='macro')
f1_val = f1_score(y_val_sent, lr_sent_pipe.predict(X_val_text), average='macro')
print(f'LR Sent -> Train F1: {f1_train:.4f} | CV F1: {f1_cv:.4f} | Eyeball F1: {f1_eyeball:.4f} | Val F1: {f1_val:.4f}')
lr_sent_scores = {'model': 'LogisticRegression', 'pipe': lr_sent_pipe, 'train_f1': f1_train, 'cv_f1': f1_cv, 'eyeball_f1': f1_eyeball, 'val_f1': f1_val}

LR Sent -> Train F1: 0.9530 | CV F1: 0.7331 | Eyeball F1: 0.7536 | Val F1: 0.7572


In [22]:
from sklearn.svm import LinearSVC
#3. LinearSVC trong Pipeline + StratifiedKFold(5) thuan tren df_train
pipe_svc = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', LinearSVC(max_iter=3000, random_state=42))])
svc_grid = {'clf__C': [0.05, 0.1, 0.2, 0.5, 1.0], 'clf__class_weight': [None, 'balanced', smooth_w_sent]}
svc_search = GridSearchCV(pipe_svc, svc_grid, cv=skf, scoring='f1_macro', n_jobs=1, refit=True, error_score='raise')  # loi MemoryError (het RAM) phai hien ro, khong nuot
svc_search.fit(X_tr_text, y_tr_sent)
svc_sent_pipe = svc_search.best_estimator_
print('SVC best params:', svc_search.best_params_)

SVC best params: {'clf__C': 0.2, 'clf__class_weight': 'balanced'}


In [23]:
f1_train = f1_score(y_tr_sent, svc_sent_pipe.predict(X_tr_text), average='macro')
f1_cv = float(svc_search.best_score_)
f1_eyeball = f1_score(y_eye_sent, svc_sent_pipe.predict(X_eye_text), average='macro')
f1_val = f1_score(y_val_sent, svc_sent_pipe.predict(X_val_text), average='macro')
print(f'SVC Sent -> Train F1: {f1_train:.4f} | CV F1: {f1_cv:.4f} | Eyeball F1: {f1_eyeball:.4f} | Val F1: {f1_val:.4f}')
svc_sent_scores = {'model': 'LinearSVC', 'pipe': svc_sent_pipe, 'train_f1': f1_train, 'cv_f1': f1_cv, 'eyeball_f1': f1_eyeball, 'val_f1': f1_val}

SVC Sent -> Train F1: 0.9257 | CV F1: 0.7178 | Eyeball F1: 0.7369 | Val F1: 0.7469


In [24]:
#Tu dong chon mo hinh co Val Dev Macro F1 cao nhat (khong gan cung)
sent_candidates = [nb_sent_scores, lr_sent_scores, svc_sent_scores]
df_sent_cmp = pd.DataFrame([{k: v for k, v in r.items() if k != 'pipe'} for r in sent_candidates])
best_sent_row = max(sent_candidates, key=lambda r: r['val_f1'])
best_sent_name, best_sent_pipe = best_sent_row['model'], best_sent_row['pipe']
print('Winner Sentiment on Val Dev:', best_sent_name, '| Val F1:', round(best_sent_row['val_f1'], 4))
df_sent_cmp

Winner Sentiment on Val Dev: LogisticRegression | Val F1: 0.7572


,model,train_f1,cv_f1,eyeball_f1,val_f1
0,ComplementNB,0.888912,0.702359,0.692405,0.724023
1,LogisticRegression,0.953023,0.733062,0.753614,0.757152
2,LinearSVC,0.925730,0.717801,0.736938,0.746857


## MÔ HÌNH ĐA KHÍA CẠNH (MULTI-LABEL ONE-VS-REST)

In [25]:
from sklearn.multiclass import OneVsRestClassifier
#Dung 5-fold phan tang theo khia canh chinh tren df_train de tune Multi-Label OneVsRestClassifier
asp_cv = list(skf.split(X_tr_text, y_tr_asp))
len(asp_cv)

5

In [26]:
#1. OneVsRestClassifier(LogisticRegression) trong Pipeline + 5-fold CV tren df_train (mo rong C theo E02)
pipe_ovr_lr = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)))])
ovr_lr_grid = {'clf__estimator__C': [0.5, 1.0, 2.0, 5.0, 10.0]}
ovr_lr_search = GridSearchCV(pipe_ovr_lr, ovr_lr_grid, cv=asp_cv, scoring='f1_macro', n_jobs=1, refit=True)
ovr_lr_search.fit(X_tr_text, Y_tr_asp)
ovr_lr_pipe = ovr_lr_search.best_estimator_
print('OVR LR best params:', ovr_lr_search.best_params_)

OVR LR best params: {'clf__estimator__C': 10.0}


In [27]:
ml_tr = f1_score(Y_tr_asp, ovr_lr_pipe.predict(X_tr_text), average='macro')
ml_cv = float(ovr_lr_search.best_score_)
ml_eye = f1_score(Y_eye_asp, ovr_lr_pipe.predict(X_eye_text), average='macro')
ml_blk = f1_score(Y_val_asp, ovr_lr_pipe.predict(X_val_text), average='macro')
blk_prim_lr = multilabel_to_primary_aspect(ovr_lr_pipe.predict_proba(X_val_text), threshold=0.5)
prim_blk_lr = f1_score(y_val_asp, blk_prim_lr, average='macro')
hit_blk_lr = float(Y_val_asp[np.arange(len(Y_val_asp)), blk_prim_lr].mean())
print(f'OVR LR -> Train ML-F1: {ml_tr:.4f} | CV ML-F1: {ml_cv:.4f} | Eyeball ML-F1: {ml_eye:.4f} | Val ML-F1: {ml_blk:.4f} | Prim-F1: {prim_blk_lr:.4f} | Hit-Any: {hit_blk_lr:.4f}')
ovr_lr_scores = {'model': 'OVR_LogisticRegression', 'pipe': ovr_lr_pipe, 'train_ml_f1': ml_tr, 'cv_ml_f1': ml_cv, 'eyeball_ml_f1': ml_eye, 'val_ml_f1': ml_blk, 'val_prim_f1': prim_blk_lr, 'val_hit_any': hit_blk_lr}

OVR LR -> Train ML-F1: 0.9930 | CV ML-F1: 0.8971 | Eyeball ML-F1: 0.8947 | Val ML-F1: 0.9017 | Prim-F1: 0.4879 | Hit-Any: 0.9536


In [28]:
#2. OneVsRestClassifier(LinearSVC) trong Pipeline + 5-fold CV tren df_train (mo rong C theo E02)
pipe_ovr_svc = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', OneVsRestClassifier(LinearSVC(max_iter=3000, class_weight='balanced', random_state=42)))])
ovr_svc_grid = {'clf__estimator__C': [0.1, 0.3, 0.5, 1.0, 2.0]}
ovr_svc_search = GridSearchCV(pipe_ovr_svc, ovr_svc_grid, cv=asp_cv, scoring='f1_macro', n_jobs=1, refit=True)
ovr_svc_search.fit(X_tr_text, Y_tr_asp)
ovr_svc_pipe = ovr_svc_search.best_estimator_
print('OVR SVC best params:', ovr_svc_search.best_params_)

OVR SVC best params: {'clf__estimator__C': 1.0}


In [29]:
ml_tr = f1_score(Y_tr_asp, ovr_svc_pipe.predict(X_tr_text), average='macro')
ml_cv = float(ovr_svc_search.best_score_)
ml_eye = f1_score(Y_eye_asp, ovr_svc_pipe.predict(X_eye_text), average='macro')
ml_blk = f1_score(Y_val_asp, ovr_svc_pipe.predict(X_val_text), average='macro')
blk_prim_svc = multilabel_to_primary_aspect(expit(ovr_svc_pipe.decision_function(X_val_text)), threshold=0.5)
prim_blk_svc = f1_score(y_val_asp, blk_prim_svc, average='macro')
hit_blk_svc = float(Y_val_asp[np.arange(len(Y_val_asp)), blk_prim_svc].mean())
print(f'OVR SVC -> Train ML-F1: {ml_tr:.4f} | CV ML-F1: {ml_cv:.4f} | Eyeball ML-F1: {ml_eye:.4f} | Val ML-F1: {ml_blk:.4f} | Prim-F1: {prim_blk_svc:.4f} | Hit-Any: {hit_blk_svc:.4f}')
ovr_svc_scores = {'model': 'OVR_LinearSVC', 'pipe': ovr_svc_pipe, 'train_ml_f1': ml_tr, 'cv_ml_f1': ml_cv, 'eyeball_ml_f1': ml_eye, 'val_ml_f1': ml_blk, 'val_prim_f1': prim_blk_svc, 'val_hit_any': hit_blk_svc}

OVR SVC -> Train ML-F1: 0.9947 | CV ML-F1: 0.8977 | Eyeball ML-F1: 0.8944 | Val ML-F1: 0.9019 | Prim-F1: 0.4717 | Hit-Any: 0.9554


In [30]:
#Tu dong chon mo hinh Multi-Label Aspect co Val Dev Multi-label Macro F1 cao nhat
asp_candidates = [ovr_lr_scores, ovr_svc_scores]
df_asp_cmp = pd.DataFrame([{k: v for k, v in r.items() if k != 'pipe'} for r in asp_candidates])
best_asp_row = max(asp_candidates, key=lambda r: r['val_ml_f1'])
best_asp_name, best_asp_pipe = best_asp_row['model'], best_asp_row['pipe']
print('Winner Aspect on Val Dev:', best_asp_name, '| Val ML-F1:', round(best_asp_row['val_ml_f1'], 4))
df_asp_cmp

Winner Aspect on Val Dev: OVR_LinearSVC | Val ML-F1: 0.9019


,model,train_ml_f1,cv_ml_f1,eyeball_ml_f1,val_ml_f1,val_prim_f1,val_hit_any
0,OVR_LogisticRegression,0.993027,0.897125,0.894679,0.901747,0.487934,0.953571
1,OVR_LinearSVC,0.994735,0.897700,0.894398,0.901900,0.471746,0.955357


In [31]:
#Tune 5 nguong rieng cho 5 khia canh bang 5-fold OOF thuan tren df_train (tu E09, zero leakage)
from sklearn.model_selection import cross_val_predict
if hasattr(best_asp_pipe, 'predict_proba'):
    oof_asp_scores = cross_val_predict(best_asp_pipe, X_tr_text, Y_tr_asp, cv=asp_cv, method='predict_proba', n_jobs=1)
    val_asp_scores = best_asp_pipe.predict_proba(X_val_text)
else:
    oof_asp_scores = expit(cross_val_predict(best_asp_pipe, X_tr_text, Y_tr_asp, cv=asp_cv, method='decision_function', n_jobs=1))
    val_asp_scores = expit(best_asp_pipe.decision_function(X_val_text))

best_asp_ths_list = []
th_grid = np.round(np.arange(0.30, 0.71, 0.02), 2)
for j in range(5):
    best_t, best_f = 0.5, -1.0
    for t in th_grid:
        f = f1_score(Y_tr_asp[:, j], (oof_asp_scores[:, j] >= t).astype(int), zero_division=0)
        if f > best_f:
            best_t, best_f = float(t), float(f)
    best_asp_ths_list.append(best_t)
best_asp_ths = np.asarray(best_asp_ths_list, dtype=float)
val_ml_05 = f1_score(Y_val_asp, (val_asp_scores >= 0.5).astype(int), average='macro')
val_ml_th = f1_score(Y_val_asp, (val_asp_scores >= best_asp_ths).astype(int), average='macro')
print('OOF per-aspect thresholds:', dict(zip(ASPECT_GROUPS, best_asp_ths_list)))
print(f'Val Multi-label F1 -> fixed 0.5: {val_ml_05:.4f} | OOF per-class thresholds: {val_ml_th:.4f}')

OOF per-aspect thresholds: {'PERF_FEAT': 0.46, 'SCREEN_CAM': 0.48, 'BATTERY': 0.46, 'PRICE_SER': 0.48, 'DESIGN_GEN': 0.36}
Val Multi-label F1 -> fixed 0.5: 0.9019 | OOF per-class thresholds: 0.9085


## PHÂN TÍCH LỖI TRÊN TẬP EYEBALL DEV (ANDREW NG)

In [32]:
#Chi soi loi tren df_eyeball (250 mau), tuyet doi khong nhin df_val (560 mau)
pred_eye_sent = best_sent_pipe.predict(X_eye_text)
df_err = df_eyeball[y_eye_sent != pred_eye_sent].copy()
df_err['pred_sent'] = pred_eye_sent[y_eye_sent != pred_eye_sent]
print('So cau doan sai tren Eyeball Dev:', len(df_err), '/', len(df_eyeball))

So cau doan sai tren Eyeball Dev: 49 / 250


In [33]:
df_err[['n_star', 'sentiment', 'pred_sent', 'sentence_clean']].head(10)

,n_star,sentiment,pred_sent,sentence_clean
898,1,1,2,"Nhân viên cửa hàng 446 Đại Mỗ, Nam Từ Liêm, Hà..."
936,2,1,0,Mua máy 15/4 mà sao chơi game thoát ra là bị đ...
287,4,0,1,Cằm dày rất nhiều so với a10. Còn về hiệu năng...
181,1,1,0,Mau hết pin quá..4500ma.mà mau hết quá tr..45%...
362,1,0,1,Hình như 16 củ thì không đáng đồng tiền bát gạ...
757,5,2,1,Điện thoại sài ngon lắm. Chơi liên quân good. ...
939,2,1,0,"Máy bị lỗi sim , đổi máy rồi vẫn bị, chụp hình..."
411,4,0,1,"Vừa mua realme c3i được hơn 1 tuần , chip medi..."
826,5,1,0,E mua con này về dùng ngon nhưng khi nâng cấp ...
295,5,2,1,Máy chụp hình rất ok không tiếc khi mua vivo s...


In [34]:
#Thu bieu thuc chinh quy bat tu noi doi lap tren 1 mau loi
tempt = df_err['sentence_clean'].iloc[0]
bool(re.search(r'(?<!\w)(?:nhưng|tuy nhiên|có điều|chỉ có|mặc dù|dù|trừ|còn lại)(?!\w)', str(tempt).lower()))

False

In [35]:
def classify_err(t):
    s = str(t).lower()
    if re.search(r'(?<!\w)(?:nhưng|tuy nhiên|có điều|chỉ có|mặc dù|dù|trừ|còn lại)(?!\w)', s): return '1. Câu đa ý trái chiều / chuyển ý (nhưng, tuy nhiên, chỉ có điều)'
    if re.search(r'(?<!\w)(?:không|chưa|chẳng|chả|đâu có)(?!\w)', s): return '2. Phủ định / đảo ngữ phức tạp'
    if len(s.split()) < 15: return '3. Câu ngắn / hàm ý thiếu từ khóa'
    return '4. Ẩn dụ / Từ vựng hiếm / Khác'
df_err['error_type'] = df_err['sentence_clean'].apply(classify_err)
df_non_err = df_eyeball[df_eyeball['sentiment'] == pred_eye_sent].copy()
non_err_type = df_non_err['sentence_clean'].apply(classify_err)
err_cnt = pd.DataFrame({
    'err_n': df_err['error_type'].value_counts(),
    'err_pct': (df_err['error_type'].value_counts(normalize=True) * 100).round(1),
    'non_err_pct': (non_err_type.value_counts(normalize=True) * 100).round(1),
}).fillna(0)
err_cnt['lift'] = (err_cnt['err_pct'] / err_cnt['non_err_pct'].replace(0, np.nan)).round(2)
err_cnt


,err_n,err_pct,non_err_pct,lift
"1. Câu đa ý trái chiều / chuyển ý (nhưng, tuy nhiên, chỉ có điều)",16.0,32.7,17.9,1.83
2. Phủ định / đảo ngữ phức tạp,15.0,30.6,39.8,0.77
3. Câu ngắn / hàm ý thiếu từ khóa,0.0,0.0,2.0,0.00
4. Ẩn dụ / Từ vựng hiếm / Khác,18.0,36.7,40.3,0.91


In [36]:
err_cnt = df_err['error_type'].value_counts()
err_pct = (err_cnt / max(len(df_err), 1) * 100).round(1)
pd.DataFrame({'so_luong_loi': err_cnt, 'ty_le_phan_tram': err_pct})

,so_luong_loi,ty_le_phan_tram
error_type,,
4. Ẩn dụ / Từ vựng hiếm / Khác,18,36.7
"1. Câu đa ý trái chiều / chuyển ý (nhưng, tuy nhiên, chỉ có điều)",16,32.7
2. Phủ định / đảo ngữ phức tạp,15,30.6


## ĐÁNH GIÁ NIÊM PHONG TRÊN TẬP TEST VÀ LƯU MÔ HÌNH

In [37]:
test_pred_sent = best_sent_pipe.predict(X_te_text)
print('Test Sentiment Macro F1:', round(f1_score(y_te_sent, test_pred_sent, average='macro'), 4))
print('Test Sentiment Accuracy:', round(accuracy_score(y_te_sent, test_pred_sent), 4))
print(classification_report(y_te_sent, test_pred_sent, target_names=['Tiêu cực', 'Trung lập', 'Tích cực'], digits=4))

Test Sentiment Macro F1: 0.729
Test Sentiment Accuracy: 0.7806
              precision    recall  f1-score   support

    Tiêu cực     0.7995    0.8651    0.8310       756
   Trung lập     0.4919    0.4874    0.4896       435
    Tích cực     0.8923    0.8422    0.8665      1033

    accuracy                         0.7806      2224
   macro avg     0.7279    0.7315    0.7290      2224
weighted avg     0.7824    0.7806    0.7807      2224



In [38]:
#Suy dien Multi-Label Aspect voi 5 nguong OOF va quy doi Primary Aspect voi fallback argmax khi < threshold
test_asp_scores = best_asp_pipe.predict_proba(X_te_text) if hasattr(best_asp_pipe, 'predict_proba') else expit(best_asp_pipe.decision_function(X_te_text))
Y_te_pred_asp = (test_asp_scores >= best_asp_ths).astype(int)
test_pred_asp = multilabel_to_primary_aspect(test_asp_scores, threshold=best_asp_ths)
test_ml_f1 = f1_score(Y_te_asp, Y_te_pred_asp, average='macro')
test_prim_f1 = f1_score(y_te_asp, test_pred_asp, average='macro')
test_hit_any = float(Y_te_asp[np.arange(len(Y_te_asp)), test_pred_asp].mean())
print(f'Test Multi-label Macro F1: {test_ml_f1:.4f} | Primary Aspect Macro F1: {test_prim_f1:.4f} | Hit-Any-True-Group: {test_hit_any:.4f}')

Test Multi-label Macro F1: 0.9067 | Primary Aspect Macro F1: 0.5091 | Hit-Any-True-Group: 0.9586


In [39]:
print('--- MULTI-LABEL ASPECT REPORT ---')
print(classification_report(Y_te_asp, Y_te_pred_asp, target_names=ASPECT_GROUPS, digits=4, zero_division=0))
print('--- PRIMARY ASPECT REPORT ---')
print(classification_report(y_te_asp, test_pred_asp, target_names=ASPECT_GROUPS, digits=4))

--- MULTI-LABEL ASPECT REPORT ---
              precision    recall  f1-score   support

   PERF_FEAT     0.8966    0.9326    0.9143      1544
  SCREEN_CAM     0.8991    0.9147    0.9068       750
     BATTERY     0.9495    0.9645    0.9569      1014
   PRICE_SER     0.8690    0.8569    0.8629       999
  DESIGN_GEN     0.8263    0.9702    0.8925      1780

   micro avg     0.8784    0.9343    0.9055      6087
   macro avg     0.8881    0.9278    0.9067      6087
weighted avg     0.8807    0.9343    0.9057      6087
 samples avg     0.8762    0.9334    0.8874      6087

--- PRIMARY ASPECT REPORT ---
              precision    recall  f1-score   support

   PERF_FEAT     0.7084    0.6094    0.6552       873
  SCREEN_CAM     0.5089    0.4043    0.4506       282
     BATTERY     0.6434    0.5223    0.5765       494
   PRICE_SER     0.4870    0.5185    0.5022       324
  DESIGN_GEN     0.2704    0.5418    0.3607       251

    accuracy                         0.5432      2224
   macro avg 

In [40]:
#Fit cac mo hinh doc lap tren X_tr_tfidf de tuong thich SHAP (NB05) va Excel (NB06)
lr_best_kw = {k.replace('clf__', ''): v for k, v in lr_search.best_params_.items()}
svc_best_kw = {k.replace('clf__', ''): v for k, v in svc_search.best_params_.items()}
lr_sent = LogisticRegression(**lr_best_kw, max_iter=1000, random_state=42).fit(X_tr_tfidf, y_tr_sent)
svc_sent = LinearSVC(**svc_best_kw, max_iter=3000, random_state=42).fit(X_tr_tfidf, y_tr_sent)
svc_asp = LinearSVC(C=ovr_svc_search.best_params_['clf__estimator__C'], class_weight='balanced', max_iter=3000, random_state=42).fit(X_tr_tfidf, y_tr_asp)

In [41]:
import json
#Luu pipeline thang giai, 5 nguong OOF va cac mo hinh tuong thich ra models/
joblib.dump(best_sent_pipe, os.path.join(MODEL_DIR, 'baseline_sentiment.pkl'))
joblib.dump(best_asp_pipe, os.path.join(MODEL_DIR, 'baseline_aspect_multilabel.pkl'))
joblib.dump(vec, os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))
joblib.dump(lr_sent, os.path.join(MODEL_DIR, 'lr_sentiment.pkl'))
joblib.dump(svc_sent, os.path.join(MODEL_DIR, 'svc_sentiment.pkl'))
joblib.dump(svc_asp, os.path.join(MODEL_DIR, 'svc_aspect.pkl'))
with open(os.path.join(MODEL_DIR, 'baseline_aspect_thresholds.json'), 'w', encoding='utf-8') as f:
    json.dump(dict(zip(ASPECT_GROUPS, [round(float(x), 4) for x in best_asp_ths])), f, indent=2, ensure_ascii=False)

In [42]:
df_baseline_pred = df_test.copy()
df_baseline_pred['pred_sentiment'] = test_pred_sent
for i in range(5):
    df_baseline_pred[f'pred_asp_{i}'] = Y_te_pred_asp[:, i]
df_baseline_pred['pred_aspect'] = test_pred_asp
out_csv = os.path.join(PROC_DIR, 'test_pred_baseline.csv')
df_baseline_pred.to_csv(out_csv, index=False, encoding='utf-8-sig')
df_baseline_pred.shape

(2224, 31)

In [43]:
sorted(os.listdir(MODEL_DIR))

['baseline_absa_5x4.pkl',
 'baseline_absa_metrics.json',
 'baseline_aspect_multilabel.pkl',
 'baseline_aspect_thresholds.json',
 'baseline_sentiment.pkl',
 'lr_aspect.pkl',
 'lr_sentiment.pkl',
 'model_metadata.json',
 'phobert_absa_v4',
 'phobert_multitask',
 'svc_aspect.pkl',
 'svc_sentiment.pkl',
 'tfidf_vectorizer.pkl',
 'training_history.json']

In [44]:
pd.read_csv(out_csv)[['sentence_clean', 'sentiment', 'pred_sentiment', 'aspect_group', 'pred_aspect', 'pred_asp_0', 'pred_asp_1', 'pred_asp_2', 'pred_asp_3', 'pred_asp_4']].head(3)

,sentence_clean,sentiment,pred_sentiment,aspect_group,pred_aspect,pred_asp_0,pred_asp_1,pred_asp_2,pred_asp_3,pred_asp_4
0,"Điện thoải ổn. Facelock cực nhanh, vân tay ôk ...",2,2,0,0,1,1,1,0,1
1,"Mình mới mua vivo91c. Tải ứng dụng ,games nhan...",2,1,0,4,1,1,0,0,1
2,Xấu đẹp gì không biết nhưng rất ưng Thế Giới D...,1,2,4,3,0,0,0,1,1


## TRUE ABSA 5x4 BASELINE (1 HỌ MÔ HÌNH ĐỒNG NHẤT CHO 5 KHÍA CẠNH)

In [45]:
#Huấn luyện 5x4 ABSA Baseline theo 1 họ mô hình đồng nhất (chọn LogReg vs LinearSVC theo Val Dev absa_macro_f1)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
Y_tr_absa = df_train[ABSA_COLS].values
Y_val_absa = df_val[ABSA_COLS].values
Y_te_absa = df_test[ABSA_COLS].values
X_val_text = df_val['sentence_seg'].fillna('')
absa_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
family_results = {}
for fam_name, make_clf in [
    ('LogisticRegression', lambda c: LogisticRegression(C=c, class_weight='balanced', max_iter=1000, random_state=42)),
    ('LinearSVC', lambda c: LinearSVC(C=c, class_weight='balanced', max_iter=3000, random_state=42)),
]:
    #Tune C dùng chung cho cả 5 đầu bằng 5-fold OOF trên train
    best_c, best_cv_f1 = 1.0, -1.0
    for c_val in [0.5, 1.0, 2.0]:
        cv_preds = []
        for j in range(5):
            p = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', make_clf(c_val))])
            cv_preds.append(cross_val_predict(p, X_tr_text, Y_tr_absa[:, j], cv=absa_cv, n_jobs=1))
        f_cv = compute_absa_metrics(Y_tr_absa, np.column_stack(cv_preds))['absa_macro_f1']
        if f_cv > best_cv_f1:
            best_c, best_cv_f1 = c_val, f_cv
    models_5 = []
    val_preds = []
    for j in range(5):
        clf = make_clf(best_c).fit(X_tr_tfidf, Y_tr_absa[:, j])
        models_5.append(clf)
        val_preds.append(clf.predict(vec.transform(X_val_text)))
    val_m = compute_absa_metrics(Y_val_absa, np.column_stack(val_preds))
    family_results[fam_name] = {'C': best_c, 'cv_absa_macro_f1': best_cv_f1, 'val_absa_macro_f1': val_m['absa_macro_f1'], 'models': models_5}
    print(f'{fam_name} (C={best_c}) -> 5-fold CV absa_macro_f1={best_cv_f1:.4f} | Val Dev absa_macro_f1={val_m["absa_macro_f1"]:.4f}')
best_absa_fam = max(family_results, key=lambda k: family_results[k]['val_absa_macro_f1'])
absa_models = family_results[best_absa_fam]['models']
print('Winner uniform 5x4 family:', best_absa_fam, 'C=', family_results[best_absa_fam]['C'])


LogisticRegression (C=2.0) -> 5-fold CV absa_macro_f1=0.7417 | Val Dev absa_macro_f1=0.7538


LinearSVC (C=0.5) -> 5-fold CV absa_macro_f1=0.7493 | Val Dev absa_macro_f1=0.7555
Winner uniform 5x4 family: LinearSVC C= 0.5


In [46]:
#Đánh giá Test ABSA 5x4 toàn bộ + lát cắt xung đột + lưu artifacts ABSA baseline
X_te_tfidf = vec.transform(X_te_text)
te_pred_absa = np.column_stack([m.predict(X_te_tfidf) for m in absa_models])
te_m_all = compute_absa_metrics(Y_te_absa, te_pred_absa)
conf_mask = df_test['has_conflict'].values.astype(bool)
conf_neg_mask = df_test['conflict_with_neg'].values.astype(bool)
te_m_conf = compute_absa_metrics(Y_te_absa[conf_mask], te_pred_absa[conf_mask])
te_m_conf_neg = compute_absa_metrics(Y_te_absa[conf_neg_mask], te_pred_absa[conf_neg_mask])
print('Test All 5x4:', te_m_all)
print('Test Conflict (n=' + str(int(conf_mask.sum())) + '):', te_m_conf)
print('Test Conflict+Neg (n=' + str(int(conf_neg_mask.sum())) + '):', te_m_conf_neg)

absa_bundle = {'family': best_absa_fam, 'C': family_results[best_absa_fam]['C'], 'models': absa_models, 'absa_cols': ABSA_COLS}
joblib.dump(absa_bundle, os.path.join(MODEL_DIR, 'baseline_absa_5x4.pkl'))
absa_metrics_out = {
    'family': best_absa_fam,
    'C': family_results[best_absa_fam]['C'],
    'test_all': te_m_all,
    'test_conflict': te_m_conf,
    'test_conflict_with_neg': te_m_conf_neg,
}
with open(os.path.join(MODEL_DIR, 'baseline_absa_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(absa_metrics_out, f, indent=2, ensure_ascii=False)

#Cập nhật test_pred_baseline.csv thêm cột pred_absa_0..4 và mức độ khiếu nại ngầm
for j in range(5):
    df_baseline_pred[f'pred_absa_{j}'] = te_pred_absa[:, j]
df_baseline_pred['pred_absa_derived_sent'] = derive_overall_sentiment(te_pred_absa)
cnt_neg, levels, is_hidden_high = classify_silent_complaint_levels(df_baseline_pred['n_star'].values, te_pred_absa)
df_baseline_pred['So_Khia_Canh_Tieu_Cuc'] = cnt_neg
df_baseline_pred['Muc_Do_Khieu_Nai_Ngam'] = levels
df_baseline_pred['Khieu_Nai_Ngam_4_5_Sao'] = is_hidden_high
df_baseline_pred.to_csv(out_csv, index=False, encoding='utf-8-sig')
print('Updated test_pred_baseline.csv with 5x4 ABSA columns:', df_baseline_pred.shape)


Test All 5x4: {'absa_micro_f1': 0.7576, 'absa_macro_f1': 0.7556, 'mean_4cls_macro_f1': 0.6895, 'per_aspect_4cls_f1': [0.6427, 0.7309, 0.7251, 0.7421, 0.6068], 'polarity_given_detected_macro_f1': 0.7854, 'majority_4cls_macro_f1': 0.1693, 'all1_presence_macro_f1': 0.6789}
Test Conflict (n=286): {'absa_micro_f1': 0.6492, 'absa_macro_f1': 0.653, 'mean_4cls_macro_f1': 0.6116, 'per_aspect_4cls_f1': [0.5678, 0.6962, 0.689, 0.6081, 0.4968], 'polarity_given_detected_macro_f1': 0.6858, 'majority_4cls_macro_f1': 0.1588, 'all1_presence_macro_f1': 0.7778}
Test Conflict+Neg (n=141): {'absa_micro_f1': 0.6311, 'absa_macro_f1': 0.6316, 'mean_4cls_macro_f1': 0.6103, 'per_aspect_4cls_f1': [0.5416, 0.709, 0.7301, 0.6233, 0.4477], 'polarity_given_detected_macro_f1': 0.6824, 'majority_4cls_macro_f1': 0.1657, 'all1_presence_macro_f1': 0.7653}


Updated test_pred_baseline.csv with 5x4 ABSA columns: (2224, 40)
